# 12주차: 설명의 한계와 프라이버시

- CPU 실행, Python 표준 라이브러리만 사용, GPU·설치·모델 호출 불필요
- 작성한 응답 예시와 상정 참여 레코드 사용. 실제 모델 출력·개인자료 없음
- 수정할 곳: 노이즈 multiplier 숫자 하나. 변경 없이 전체 실행 가능
- 공개 제주 문서 원문·저자 목록: https://ko.wikipedia.org/w/index.php?oldid=42464087
- 원문 CC BY-SA 4.0, 실행 시 출처와 판 번호 출력, 원문 저장 없음
- 논문: https://arxiv.org/abs/2305.04388
- DP 원리: https://www.cis.upenn.edu/~aaroth/Papers/privacybook.pdf 2.3, 3.3, 3.5절


## 1. 공개 문서의 출처 확인

- 행정시 이름을 공개 문서에서 확인. 응답 예시는 원문 인용이 아닌 작성 예시
- 네트워크 실패는 미확인으로 출력. 예시 실행이 성공해도 공개 사실 확인 성공을 주장하지 않음


In [1]:
import html, json, math, random, re, statistics, urllib.parse, urllib.request
OLDID = 42464087
SOURCE = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
print("공개 문서 및 저자 목록:", SOURCE, "라이선스: CC BY-SA 4.0")
source_verified = False
api = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
    "action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
try:
    request = urllib.request.Request(api, headers={"User-Agent": "deepnlp-course/1.0 (yj.lee@chu.ac.kr)"})
    with urllib.request.urlopen(request, timeout=30) as response:
        parsed = json.load(response)["parse"]
    assert parsed["revid"] == OLDID, "판 번호 불일치"
    source_text = html.unescape(re.sub(r"<[^>]+>", " ", parsed["text"]))
    assert "제주시" in source_text and "서귀포시" in source_text, "범주 이름 확인 실패"
    source_verified = True
    print("원문 판 번호 확인:", parsed["revid"], "범주 이름: 제주시, 서귀포시")
    del source_text, parsed
except (OSError, ValueError, KeyError, AssertionError) as error:
    print("공개 문서 조회 미확인:", type(error).__name__)
print("이후 자료: 교수 작성 응답과 상정 레코드, 실제 모델·개인 응답 아님")


공개 문서 및 저자 목록: https://ko.wikipedia.org/w/index.php?oldid=42464087 라이선스: CC BY-SA 4.0


원문 판 번호 확인: 42464087 범주 이름: 제주시, 서귀포시
이후 자료: 교수 작성 응답과 상정 레코드, 실제 모델·개인 응답 아님


## 2. 서술-결과 불일치 프록시

- 설명에서 주장한 수와 최종 답의 수만 비교하는 좁은 규칙
- reference_count=2는 교육 예제의 정답 기준, 최신 행정 변화를 추정하는 모델 아님
- 내부 계산 접근 없음. 모든 예시의 내부 충실성은 미측정


In [2]:
reference_count = 2
examples = [
    {"id": "A", "reason_text": "제주시와 서귀포시, 행정시 수는 2.", "reason_count": 2, "answer_count": 2},
    {"id": "B", "reason_text": "행정시 수는 3.", "reason_count": 3, "answer_count": 2},
    {"id": "C", "reason_text": "제주시와 서귀포시, 행정시 수는 2.", "reason_count": 2, "answer_count": 3},
    {"id": "D", "reason_text": "행정시 수는 3.", "reason_count": 3, "answer_count": 3},
]
print("작성 예시 | 설명 수 | 답 수 | 설명 수 정확 | 답 정확 | 불일치 | 내부 충실성")
for e in examples:
    e["mismatch"] = e["reason_count"] != e["answer_count"]
    print(e["id"], e["reason_count"], e["answer_count"],
          e["reason_count"] == reference_count, e["answer_count"] == reference_count,
          e["mismatch"], "미측정", sep=" | ")
print("불일치:", sum(e["mismatch"] for e in examples), "/", len(examples))
print("B: 정답이어도 설명 불일치. D: 불일치 없어도 오답. 내부 faithfulness 측정값 아님.")


작성 예시 | 설명 수 | 답 수 | 설명 수 정확 | 답 정확 | 불일치 | 내부 충실성
A | 2 | 2 | True | True | False | 미측정
B | 3 | 2 | False | True | True | 미측정
C | 2 | 3 | True | False | True | 미측정
D | 3 | 3 | False | False | False | 미측정
불일치: 2 / 4
B: 정답이어도 설명 불일치. D: 불일치 없어도 오답. 내부 faithfulness 측정값 아님.


## 3. 상정 통계와 민감도

- 공개 행정시 이름을 범주로 사용하는 작성 선호 레코드 20개
- 실제 관광 조사, 주민 의견, 학생 정보 아님
- 인접 관계: 레코드 하나 추가/제거(add/remove), 한 명의 기여 최대 1
- 공개 질의: 제주시를 선택한 count 하나. 전역 민감도 Δf=1


In [3]:
records = ["제주시"] * 12 + ["서귀포시"] * 8
true_count = sum(place == "제주시" for place in records)
sensitivity = 1.0
base_epsilon = 1.0
print("상정 레코드 수:", len(records), "제주시 count:", true_count)
for added in ("제주시", "서귀포시"):
    neighbor = records + [added]
    neighbor_count = sum(place == "제주시" for place in neighbor)
    print("하나 추가:", added, "count 변화:", abs(neighbor_count - true_count))
print("한 레코드의 0/1 기여 제한으로 민감도 1. 실제 참여 자료 없음.")


상정 레코드 수: 20 제주시 count: 12
하나 추가: 제주시 count 변화: 1
하나 추가: 서귀포시 count 변화: 0
한 레코드의 0/1 기여 제한으로 민감도 1. 실제 참여 자료 없음.


## 4. 한 숫자 변경

- 먼저 기본값 그대로 전체 실행. 다음에 1.0을 2.0 또는 3.0으로 변경하고 전체 실행
- scale = multiplier × Δf / base_epsilon
- 이상적인 메커니즘의 유효 epsilon = base_epsilon / multiplier
- 공개 seed는 재현성용. 실제 민감한 데이터의 DP 공개에는 사용할 수 없음


In [4]:
# TODO: 이 숫자만 2.0 또는 3.0으로 변경
NOISE_MULTIPLIER = 3.0


In [5]:
assert math.isfinite(NOISE_MULTIPLIER) and NOISE_MULTIPLIER > 0, "양의 유한한 숫자 사용"
scale = NOISE_MULTIPLIER * sensitivity / base_epsilon
effective_epsilon = sensitivity / scale
print("scale:", scale, "이상적 메커니즘 유효 epsilon:", effective_epsilon)
print("보장 범위 주의: 공개 seed 교육 실험, 실제 DP 공개 보장 없음")


scale: 3.0 이상적 메커니즘 유효 epsilon: 0.3333333333333333
보장 범위 주의: 공개 seed 교육 실험, 실제 DP 공개 보장 없음


## 5. 같은 표본으로 scale 변화 관찰

- 표준 Laplace 표본 z에 scale b를 곱해 noise=bz 생성
- 같은 z로 scale만 비교하므로 평균 절대 오차는 b에 정확히 비례
- 이론적 E|noise|=b, 표본 평균은 근사값
- 음수를 0으로 바꾸는 clipping은 공개 출력만 쓰는 사후 처리
- 이 노트북의 2,000회 반복은 상정 데이터 오차 관찰. 실제 개인 통계를 2,000회 공개하는 계획 아님


In [6]:
rng = random.Random(202612)
def standard_laplace():
    # 열린 구간의 교육용 inverse-CDF 표본. 운영용 안전한 DP 구현 아님.
    u = 0.0
    while u == 0.0:
        u = rng.random()
    return math.log(2 * u) if u < 0.5 else -math.log(2 * (1 - u))

standard_noise = [standard_laplace() for _ in range(2000)]
print("multiplier | scale | 유효 epsilon | 평균절대오차 | 음수출력비율 | clipping후오차")
for multiplier in dict.fromkeys([1.0, NOISE_MULTIPLIER]):
    b = multiplier * sensitivity / base_epsilon
    released = [true_count + b * z for z in standard_noise]
    mae = statistics.mean(abs(value - true_count) for value in released)
    clipped = [max(0.0, value) for value in released]
    clipped_mae = statistics.mean(abs(value - true_count) for value in clipped)
    negative_rate = sum(value < 0 for value in released) / len(released)
    print(f"{multiplier:.1f} | {b:.1f} | {sensitivity / b:.3f} | {mae:.3f} | {negative_rate:.3f} | {clipped_mae:.3f}")
print("표준표본 평균 절댓값:", round(statistics.mean(abs(z) for z in standard_noise), 6))
print("같은 표본 사용: 변경 전후 clipping 전 MAE 비율 = multiplier 비율")


multiplier | scale | 유효 epsilon | 평균절대오차 | 음수출력비율 | clipping후오차
1.0 | 1.0 | 1.000 | 0.969 | 0.000 | 0.969
3.0 | 3.0 | 0.333 | 2.906 | 0.006 | 2.891
표준표본 평균 절댓값: 0.96873
같은 표본 사용: 변경 전후 clipping 전 MAE 비율 = multiplier 비율


## 6. 반복 공개와 판단 보류

- 이상적인 epsilon-DP 독립 공개 k회의 기본 순차 합성 상한: k × epsilon
- 여러 응답의 평균을 공개한다고 예산이 사라지지 않음
- 공개 seed로 계산한 아래 숫자는 예산 산술 관찰이며 보호된 서비스가 아님
- 실제 프로젝트는 노이즈를 더하기 전에 수집하지 않을 자료와 사람 확인 경계를 먼저 설정


In [7]:
for k in (1, 5, 10):
    print("공개 횟수:", k, "기본 순차 합성 epsilon 상한:", round(k * effective_epsilon, 3))
print("허용 예시: 공개 문서 출처와 확인 시점 안내")
print("보류 예시: 안전·자격·권리 최종 판정, 지역 정책 수용성 자동 확정")
print("출처 확인 상태:", "확인" if source_verified else "미확인")
print("완료: 내부 충실성 미측정, 실제 개인정보 없음, 실제 DP 서비스 보장 없음")


공개 횟수: 1 기본 순차 합성 epsilon 상한: 0.333
공개 횟수: 5 기본 순차 합성 epsilon 상한: 1.667
공개 횟수: 10 기본 순차 합성 epsilon 상한: 3.333
허용 예시: 공개 문서 출처와 확인 시점 안내
보류 예시: 안전·자격·권리 최종 판정, 지역 정책 수용성 자동 확정
출처 확인 상태: 확인
완료: 내부 충실성 미측정, 실제 개인정보 없음, 실제 DP 서비스 보장 없음


## 결과 해석 기록

- B와 D의 차이: [불일치 여부]와 [정답 여부]를 다른 문장으로 설명
- multiplier [값], scale [값], 유효 epsilon [값], 관찰 MAE [값]
- 공개 seed·실제 기여 제한·반복 공개 예산 중 운영 전 바꿀 조건 두 가지 기록
- 강의의 위험 체크리스트와 설계도 §선 양식을 팀 목적에 맞게 작성


## 돌아보기: 확인 질문

1. B와 D의 설명-결과 불일치와 최종 답 정확성을 서로 다른 문장으로 기록
2. multiplier 1과 3의 scale·유효 epsilon·관찰 MAE를 출력과 대조
3. 공개 seed의 재현 가능한 교육 실험이 실제 DP 공개 보장이 아닌 이유 설명
4. 프로젝트 체크리스트에서 사용 목적·사람 확인·미확인 근거를 구분

## 제출 안내

- 코드 구현이나 자료 수집을 새로 요구하는 과제가 아닌 제공된 실습의 관찰 기록
- 별도 제출 공지가 있을 때 실행한 노트북의 출력 표와 질문별 관찰·판정 이유를 함께 제출
- 저장된 결과에서 미확인·미판정은 그대로 표시, 모델 학습 성과로 보고하지 않는 원칙
- 학생 폴더에는 학번만 사용, 이름·연락처·데이터 원본·API 키 제외


## 답: 관찰 기록

- 실행: 2026-11-21 로컬 CPU(macOS, Python 3.13.12, 표준 라이브러리만 사용). 안내대로 먼저 `NOISE_MULTIPLIER = 1.0` 그대로 전체 실행하고, 그다음 `3.0`으로 바꿔 다시 전체 실행했다. 두 번 모두 오류 없이 끝났다. 저장된 출력은 3.0 실행 결과이고, 1.0 실행에서만 나온 줄(4절 scale, 6절 합성 상한)은 아래에 옮겨 적었다.
- 응답 예시와 레코드 20개는 교수 작성·상정 자료다. 아래 숫자는 공개 seed 교육 실험의 관찰값이며 실제 모델 출력, 실제 개인정보, 실제 DP 공개 보장이 아니다.

### 1절 출처 확인

- 고정판 42464087 조회에 성공했다: "원문 판 번호 확인: 42464087 범주 이름: 제주시, 서귀포시". 6절의 출처 확인 상태도 `확인`이다.
- 확인한 것은 판 번호와 두 행정시 이름이 원문에 있다는 것까지다. 응답 예시 A~D의 문장은 원문 인용이 아니다.

### 관찰 기록표

| 실험 | 출력 | 해석 |
|---|---|---|
| 서술-결과 프록시 | A: 불일치 False·답 정확 True / B: 불일치 True·답 정확 True / C: 불일치 True·답 정확 False / D: 불일치 False·답 정확 False. 불일치 2/4. 내부 충실성은 네 행 모두 `미측정` | 불일치 여부와 답 정확 여부가 서로 독립적으로 갈렸다. 두 열 중 하나만 보면 B와 D를 잘못 판정한다. |
| 민감도 | 레코드 20개, 제주시 count 12. 제주시 하나 추가 시 변화 1, 서귀포시 하나 추가 시 변화 0 | 한 사람의 기여가 최대 1이라서 Δf = 1이다. |
| multiplier 1.0 | scale 1.0, 유효 epsilon 1.000, MAE 0.969, 음수 출력 비율 0.000, clipping 후 오차 0.969 | 이론적 E\|noise\| = b = 1에 가까운 표본 평균(표준표본 평균 절댓값 0.96873) |
| multiplier 3.0 | scale 3.0, 유효 epsilon 0.333, MAE 2.906, 음수 출력 비율 0.006, clipping 후 오차 2.891 | 같은 표본이라 MAE가 정확히 3배다(0.96873 × 3 = 2.906). 노이즈가 커져 음수 출력 12건(0.6%)이 생겼고, 0으로 clipping하자 오차가 2.906 → 2.891로 조금 줄었다. |
| 반복 공개 | 1.0: 1·5·10회 상한 1.0·5.0·10.0 / 3.0: 0.333·1.667·3.333 | 공개 횟수에 비례해 상한이 커진다. multiplier를 3배로 해도 10회 공개하면 상한 3.333으로, 1.0의 1회 공개(1.0)보다 크다. |

식과 대조: scale = multiplier × Δf / base_epsilon으로 계산한 값(scale 3, 유효 epsilon 1/3, 같은 표본이므로 MAE 3배)과 출력이 일치했다.

### 결과 해석 기록

- B와 D의 차이: B는 설명의 수(3)와 답의 수(2)가 달라 **불일치가 있다**. 그와 별개로 B의 답 2는 기준값 2와 같아 **답은 맞았다**. D는 설명과 답이 모두 3이라 **불일치가 없다**. 그와 별개로 D의 답 3은 기준값 2와 달라 **답은 틀렸다**.
- multiplier 3.0, scale 3.0, 유효 epsilon 0.333, 관찰 MAE 2.906(기준 1.0일 때 scale 1.0, epsilon 1.000, MAE 0.969).
- 운영 전 바꿀 조건 두 가지:
  1. **공개 seed**: seed 202612가 노트북에 공개돼 있어 같은 노이즈를 누구나 다시 만들 수 있다. 운영에서는 공개되지 않는 안전한 난수원과 검증된 DP 구현으로 바꿔야 한다.
  2. **반복 공개 예산**: 공개 횟수만큼 epsilon 상한이 쌓인다(3.0에서도 10회면 3.333). 운영 전에 전체 예산과 최대 공개 횟수를 정하고, 예산을 다 쓰면 공개를 멈춰야 한다.

### 위험 체크리스트(팀 프로젝트: fewfew 2.0)

fewfew 2.0은 모바일 앱에서 알고리즘 문제를 받고, 폰에서 코드를 쓰고, 폰에서 바로 채점받는 개인 맞춤형 문제집이다. 문제는 사용자가 고른 주제×난이도×언어로 생성 모델이 만들거나 문제 풀에서 가져오고, 채점은 기기 안의 5개 언어 실행 엔진이 오프라인으로 한다. 아래는 팀 기획 문서(2026-09-09판) 기준의 수업용 초기 검토이고, 법률 분류를 확정한 결과가 아니다.

| 검토 항목 | 팀 기입칸 | 구분 |
|---|---|---|
| 사용 목적과 사용자 | 모바일로 코딩테스트·알고리즘을 연습하려는 개인 학습자의 자기 연습 | 사용 목적 |
| 입력 자료와 기준 시점 | 생성 모델이 만든 문제, 문제 풀·내장 시작 문제, 사용자가 쓴 코드, 풀이 기록으로 만든 구조화 프로필(약점 주제). 기획 문서 2026-09-09판 기준 | 사용 목적 |
| 결과를 사용하는 결정 | 연습 문제의 통과/실패 표시와 다음 문제 추천까지. 채용·학점·자격 인증 등 사람에 대한 최종 평가에 사용 금지 | 사용 목적 |
| 적용 관할·EU 관련성 | 미확인. 앱스토어 배포 국가에 EU가 포함되는지 운영 전 Article 2로 검토 | 미확인 근거 |
| 금지 관행 가능성 | 기획상 리그·XP·보상을 두지 않고, 순위표는 주간 리셋·보상 없음. Article 5 원문 대조는 미실시 | 미확인 근거 |
| Annex I/III 용도 해당성 | 교육기관의 입학·학습 성과 평가나 채용 선발 용도가 아닌 개인 연습. 기업 채용 코딩테스트·학교 평가에 쓰이게 되면 Annex III(교육·고용) 재검토 | 사용 목적 |
| 결정에 실질적 영향 | 통과 판정은 기기 자가 보고라서 조작될 수 있다. 그래서 순위·비교는 판돈 낮은 형태(돈·보상 없음)로 제한 | 사용 목적 |
| 자연인 프로파일링 | 있음. 풀이 기록으로 약점 주제를 추정해 맞춤 문제를 추천한다. 평가·채용 용도로 바뀌면 Article 6(3) 예외를 주장할 수 없는 조건 | 사용 목적 |
| Article 6(3) 예외 주장 | 보류. 현재 용도가 Annex III 해당이 아니라고 보지만 검토가 충분하지 않아 예외를 확정하지 않음 | 미확인 근거 |
| 사람 검토·중단 권한 | 생성 문제는 기기 사전 검사를 통과한 것만 낸다(자동 검증). 사람 검토는 운영자가 조작 의심 제출을 사후 검증·제외하는 것까지 정해졌고, 생성 문제를 사람이 내리는 절차는 미정 | 사람 확인 |
| 투명성·이의 제기 | 생성 모델이 만든 문제라는 표시가 필요(Article 50 의무 후보). 문제 신고 기능은 아직 미구현이라 운영 전 필요 | 사람 확인 |
| 수집 최소화·보관 | 자유 텍스트 조건 입력은 받지 않는다. 계정 전에는 무작위 기기 UUID만 쓴다. 계정 도입 후 제출 코드·시간을 서버에 저장할 계획이라 수집 동의·연령 확인·보관 기간을 그때 정함(보관 기간 미정) | 미확인 근거 |
| 잠정 결론·미확인 | 개인 연습 용도에서는 Annex III 해당 용도가 없다고 잠정 판단. 생성 표시 의무 후보, 신고 경로, 관할, 보관 기간이 미확인. 고위험 최종 법률 판단은 미실시 | 미확인 근거 |

### 설계도 §선(fewfew 2.0)

| 항목 | 문장 |
|---|---|
| 허용 출력 | 시스템은 사용자가 고른 주제·난이도·언어로 검증을 통과한 문제를 주고, 기기에서 실행한 테스트의 통과 여부와 같은 문제를 푼 사람들의 비교 통계를 보여 준다. |
| 넘기지 않을 판단 | 실력 인증, 채용 적합성, 학점·자격처럼 사람에 대한 최종 평가는 하지 않는다. 순위와 통과율은 연습 동기용 참고값이다. |
| 보류 조건 | 기기 사전 검사를 통과하지 못한 생성 문제는 내지 않고 문제 풀·내장 문제로 대체한다. 신고가 들어온 문제는 검토 전까지 풀에서 보류한다. |
| 확인 주체와 권한 | 운영자가 신고된 문제와 조작 의심 제출을 검토해 수정·삭제·집계 제외를 한다. |
| 이의 제기 | 이용자는 문제 화면의 신고로 문제 오류·채점 오류 재검토를 요청할 수 있다(운영 전 구현 필요). |
| 데이터 경계 | 자유 텍스트 입력과 실명은 받지 않는다. 문제별 통과율·풀이 시간 비교를 공개할 때는 사람당 문제별 첫 제출 하나만 집계해 기여를 1로 제한하고, 푼 사람이 적은 문제는 비교를 숨기는 기준과 반복 공개 예산을 먼저 정한다. |
| 재검토 계기 | 채용·교육기관 평가 용도 제휴, 계정과 제출 기록 서버 저장 시작, 튜터·해설 기능 추가, EU 배포, 생성 문제 오류 신고 급증 시 위험 검토를 갱신한다. |

사람 확인 없이 외부로 내보내지 않을 출력: 사용자의 실력을 인증하거나 채용·평가에 쓰일 수 있는 판정 문장(예: "이 사용자는 코딩테스트 합격 수준").

이번 주 실습과의 연결: 경쟁 기능의 "이 문제를 푼 n명 중 상위 k%", "첫 제출 통과율 m%"는 노트북 3~6절의 집계 공개와 같은 구조다. 한 사람이 같은 문제를 여러 번 제출하면 기여가 1을 넘어 민감도 가정(Δf = 1)이 깨진다. n이 작은 문제는 통계 하나로 특정 사용자의 결과가 드러날 수 있다. 같은 통계를 매주 다시 공개하면 순차 합성으로 예산이 쌓인다. 노이즈 방식은 아직 정하지 않았고, 기여 제한과 공개 기준부터 정하는 순서로 기록했다.

### 돌아보기 답

**1.** B의 설명은 행정시 수를 3이라 했지만 최종 답은 2라서, 설명과 결과가 **불일치**한다(출력 `불일치 True`). 이와 별개로 B의 최종 답 2는 기준값 2와 같아서 **정답**이다(`답 정확 True`). D는 설명과 최종 답이 모두 3이라 **불일치가 없다**(`불일치 False`). 이와 별개로 D의 최종 답 3은 기준값과 달라 **오답**이다(`답 정확 False`). 두 예시 모두 내부 충실성(faithfulness)은 측정하지 않았다(`미측정`). 그래서 "불일치 없음"을 "설명이 내부 계산을 반영함"으로 읽을 수 없다.

**2.** multiplier 1.0: scale 1.0, 유효 epsilon 1.000, MAE 0.969. multiplier 3.0: scale 3.0, 유효 epsilon 0.333, MAE 2.906(5절 표). 식 scale = multiplier × Δf / base_epsilon = 3 × 1 / 1 = 3, 유효 epsilon = 1 / 3 = 0.333과 일치한다. 같은 표준표본 2,000개를 쓰므로 MAE 비율은 정확히 3이다(2.906 ≈ 3 × 0.96873). 이론값 E\|noise\| = b(1과 3)와 비교하면 표본 평균이 약간 작은 근사값이다. 3.0에서는 음수 출력 비율 0.006이 생겼고, clipping 후 오차는 2.891이다.

**3.** 첫째, seed 202612가 공개돼 있어서 노이즈를 예측할 수 있다. 노트북 밖에서 같은 seed로 계산해 보니 첫 표준표본은 z₁ = −1.1436이었다. multiplier 3.0의 첫 공개값 12 + 3 × z₁ = 8.569에서 3 × z₁을 빼면 참값 12가 그대로 복원된다. DP는 공격자가 노이즈 값을 모른다는 전제 위에 서 있으므로, 재현 가능한 seed는 보호를 없앤다. 둘째, 노트북의 Laplace 표본은 셀 주석대로 교육용 inverse-CDF 구현이고 운영용 안전 구현이 아니다. 셋째, 2,000회 반복은 오차 관찰용이다. 같은 자료를 실제로 여러 번 공개하면 순차 합성으로 예산이 쌓인다(3.0에서도 10회면 상한 3.333). 넷째, 레코드는 상정 자료이고 기여 제한 1은 가정일 뿐, 실제 수집 과정에서 검증하지 않았다. 따라서 재현 가능한 교육 실험의 출력은 실제 DP 공개 보장이 아니다.

**4.** 위 fewfew 체크리스트의 "구분" 열로 나눴다. **사용 목적**은 개인 학습자의 자기 연습, 통과 표시와 추천까지만 쓰고 채용·학점·자격 평가에는 쓰지 않는 범위, 풀이 기록으로 약점 주제를 추정하는 프로파일링(있음), 판돈 낮은 비교다. **사람 확인**은 운영자가 조작 의심 제출을 사후 검증·제외하는 것(정해짐)과, 생성 문제를 사람이 내리는 절차·신고 경로(미정·미구현)다. **미확인 근거**는 EU 관할 관련성, Article 5 원문 대조, Article 6(3) 예외 주장(보류), 제출 코드 보관 기간, 고위험 최종 법률 판단이다. 기기 사전 검사 같은 자동 검증은 사람 확인으로 세지 않았고, 미확인 항목을 "해당 없음"으로 적지 않았다.
